In [ ]:
import fastf1
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

fastf1.Cache.enable_cache("cache/")

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", 20)

In [ ]:
laps = pd.read_parquet("data/raw/laps_2023_r01.parquet")
print(laps.shape)
laps.head()

In [ ]:
print("Missing values:")
print(laps.isnull().sum())
print(f"\nCompounds present: {laps['Compound'].unique()}")
print(f"\nLaps per compound:")
print(laps['Compound'].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
compounds = laps[laps['Compound'].isin(['SOFT','MEDIUM','HARD'])]
compounds = compounds[compounds['LapTimeSeconds'].between(90, 115)]  # remove outliers

sns.boxplot(data=compounds, x='Compound', y='LapTimeSeconds', 
            order=['SOFT','MEDIUM','HARD'], ax=ax)
ax.set_title('Lap time by compound — Bahrain 2023')
ax.set_ylabel('Lap time (seconds)')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for compound, color in [('SOFT','red'), ('MEDIUM','yellow'), ('HARD','white')]:
    subset = laps[
        (laps['Compound'] == compound) & 
        (laps['LapTimeSeconds'].between(90, 115)) &
        (laps['TyreLife'] <= 40)
    ]
    if len(subset) == 0:
        continue
    # Average lap time per tyre age
    avg = subset.groupby('TyreLife')['LapTimeSeconds'].median()
    ax.plot(avg.index, avg.values, label=compound, color=color, linewidth=2)

ax.set_title('Tyre degradation — Bahrain 2023')
ax.set_xlabel('Tyre age (laps)')
ax.set_ylabel('Median lap time (seconds)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
top_drivers = laps['Driver'].value_counts().head(5).index

fig, ax = plt.subplots(figsize=(12, 5))
for driver in top_drivers:
    d = laps[
        (laps['Driver'] == driver) & 
        (laps['LapTimeSeconds'].between(90, 115))
    ]
    ax.plot(d['LapNumber'], d['LapTimeSeconds'], label=driver, alpha=0.8)

ax.set_title('Lap time progression — top 5 drivers')
ax.set_xlabel('Lap number')
ax.set_ylabel('Lap time (seconds)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
findings = """
Key findings — Bahrain 2023
===========================
1. Soft tyres are X seconds faster than mediums on lap 1 but degrade by ~Xs over 20 laps
2. Hard tyres show almost flat degradation after lap 5
3. X% of laps are missing compound data — need to handle in features.py
4. Pit laps are clear outliers (>110s) — must be filtered before training
5. Track temp correlates with ... (check weather merge next)
"""
print(findings)